# SegFormer-B0 Transfer Learning: CHASEDB1 -> DRIVE

This notebook trains SegFormer-B0 with a true source-to-target transfer workflow:

1. Clone/import the project code.
2. Load the processed CHASEDB1 dataset as the source domain.
3. Load the processed DRIVE dataset as the target domain.
4. Initialize SegFormer-B0 from pretrained weights.
5. Train on CHASEDB1 and save the best source checkpoint.
6. Load the CHASEDB1 checkpoint into a fresh SegFormer-B0 model.
7. Fine-tune on DRIVE with smaller learning rates.
8. Save validation/test metrics, the final DRIVE checkpoint, and visual prediction samples.


In [ ]:
!pip -q install transformers timm albumentations opencv-python tqdm pandas matplotlib

In [ ]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/KhanhGiauTen/DL-retinal-vessel-segmentation-final.git"
REPO_DIR = Path("/kaggle/working/DL-retinal-vessel-segmentation-final")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Using repo:", REPO_DIR)
print("CWD:", Path.cwd())

In [ ]:
import gc
import json
import random
import shutil
import zipfile
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from src.models.segformer import SegFormerB0


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = torch.cuda.is_available()
print("Device:", DEVICE)
print("AMP:", AMP)

## 1. Configuration

Update the Kaggle dataset paths if your uploaded dataset slug is different. The resolver also searches common zip names under `/kaggle/input` and the repo `dataset/` folder.


In [ ]:
WORK_DIR = Path("/kaggle/working")
ARTIFACT_DIR = WORK_DIR / "segformer_b0_chase_to_drive"
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
HISTORY_DIR = ARTIFACT_DIR / "history"
RESULTS_DIR = ARTIFACT_DIR / "results"
PREDICTIONS_DIR = ARTIFACT_DIR / "predictions"

for path in (ARTIFACT_DIR, CHECKPOINT_DIR, HISTORY_DIR, RESULTS_DIR, PREDICTIONS_DIR):
    path.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 2
WEIGHT_DECAY = 1e-4
BEST_METRIC = "dice"
THRESHOLD = 0.5

# SOURCE: CHASEDB1
CHASE_DATA_DIRS = [
    Path("/kaggle/input/datasets/ngchnhphong/dl-src-transfer-dataset"),
    Path("/kaggle/input/datasets/khnhnguyn222/chasedb1-processed-dataset"),
    Path("/kaggle/input/chasedb1-processed-dataset"),
    REPO_DIR / "dataset",
]
CHASE_ZIP_NAMES = [
    "CHASEDB1_processed_dataset.zip",
    "CHASE_processed_dataset.zip",
    "chasedb1_processed_dataset.zip",
]
CHASE_EXTRACT_DIR = WORK_DIR / "CHASEDB1_processed_dataset"
CHASE_NUM_EPOCHS = 40
CHASE_PATIENCE = 10
CHASE_LR_ENCODER = 1e-5
CHASE_LR_HEAD = 1e-4
CHASE_BEST_MODEL_PATH = CHECKPOINT_DIR / "best_segformer_b0_chase.pth"
CHASE_HISTORY_PATH = HISTORY_DIR / "segformer_b0_chase_history.csv"
CHASE_RESULTS_PATH = RESULTS_DIR / "segformer_b0_chase_results.csv"

# TARGET: DRIVE
DRIVE_DATA_DIRS = [
    Path("/kaggle/input/datasets/ngchnhphong/dl-src-transfer-dataset"),
    Path("/kaggle/input/datasets/khnhnguyn222/drive-processed-dataset"),
    Path("/kaggle/input/drive-processed-dataset"),
    REPO_DIR / "dataset",
]
DRIVE_ZIP_NAMES = [
    "DRIVE_processed_dataset.zip",
    "drive_processed_dataset.zip",
]
DRIVE_EXTRACT_DIR = WORK_DIR / "DRIVE_processed_dataset"
DRIVE_NUM_EPOCHS = 50
DRIVE_PATIENCE = 12
DRIVE_LR_ENCODER = 5e-6
DRIVE_LR_HEAD = 5e-5
DRIVE_BEST_MODEL_PATH = CHECKPOINT_DIR / "best_segformer_b0_drive.pth"
FINAL_BEST_MODEL_PATH = WORK_DIR / "best_segformer_b0.pth"
DRIVE_HISTORY_PATH = HISTORY_DIR / "segformer_b0_drive_history.csv"
DRIVE_RESULTS_PATH = RESULTS_DIR / "segformer_b0_drive_results.csv"
DRIVE_PREDICTIONS_DIR = PREDICTIONS_DIR / "drive_test"

print("Artifacts will be saved to:", ARTIFACT_DIR)

## 2. Dataset Resolution

The processed datasets must contain split folders with `.pt` samples:

```text
DATA_ROOT/
  train/*.pt
  val/*.pt
  test/*.pt
```

CHASEDB1 samples may use `manual_1`; DRIVE samples may use `manual`. The dataset class below supports both.


In [ ]:
def find_existing_zip(data_dirs: list[Path], zip_names: list[str]) -> Path | None:
    for data_dir in data_dirs:
        for zip_name in zip_names:
            candidate = data_dir / zip_name
            if candidate.exists():
                return candidate

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for zip_name in zip_names:
            matches = list(kaggle_input.rglob(zip_name))
            if matches:
                return matches[0]
    return None


def find_split_root(root: Path) -> Path | None:
    if all((root / split).exists() for split in ("train", "val", "test")):
        return root
    if root.exists():
        for child in root.iterdir():
            if child.is_dir() and all((child / split).exists() for split in ("train", "val", "test")):
                return child
    return None


def resolve_processed_dataset_root(
    data_dirs: list[Path],
    zip_names: list[str],
    extract_dir: Path,
) -> Path:
    zip_path = find_existing_zip(data_dirs, zip_names)
    if zip_path is not None:
        if not find_split_root(extract_dir):
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_dir)
        split_root = find_split_root(extract_dir)
        if split_root is not None:
            print(f"Resolved {zip_path.name} -> {split_root}")
            return split_root

    for data_dir in data_dirs:
        split_root = find_split_root(data_dir)
        if split_root is not None:
            print(f"Resolved existing dataset root -> {split_root}")
            return split_root

    raise FileNotFoundError(
        "Could not find processed dataset. Checked dirs: "
        f"{[str(p) for p in data_dirs]} and zip names: {zip_names}"
    )


CHASE_ROOT = resolve_processed_dataset_root(CHASE_DATA_DIRS, CHASE_ZIP_NAMES, CHASE_EXTRACT_DIR)
DRIVE_ROOT = resolve_processed_dataset_root(DRIVE_DATA_DIRS, DRIVE_ZIP_NAMES, DRIVE_EXTRACT_DIR)

print("CHASE_ROOT:", CHASE_ROOT)
print("CHASE train files:", len(list((CHASE_ROOT / "train").glob("*.pt"))))
print("DRIVE_ROOT:", DRIVE_ROOT)
print("DRIVE train files:", len(list((DRIVE_ROOT / "train").glob("*.pt"))))

In [ ]:
IMAGENET_MEAN_TENSOR = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
IMAGENET_STD_TENSOR = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)


def prepare_segformer_image(image: torch.Tensor) -> torch.Tensor:
    image = image.float()
    if image.dim() == 3 and image.shape[0] != 3 and image.shape[-1] == 3:
        image = image.permute(2, 0, 1)
    if image.dim() != 3 or image.shape[0] != 3:
        raise ValueError(f"Expected image [3, H, W], got {tuple(image.shape)}")

    min_value = float(image.min())
    max_value = float(image.max())
    if min_value >= 0.0 and max_value > 1.0:
        image = image / 255.0
        image = (image - IMAGENET_MEAN_TENSOR) / IMAGENET_STD_TENSOR
    elif min_value >= 0.0 and max_value <= 1.0:
        image = (image - IMAGENET_MEAN_TENSOR) / IMAGENET_STD_TENSOR

    return image


def prepare_binary_mask(mask: torch.Tensor) -> torch.Tensor:
    if not torch.is_tensor(mask):
        mask = torch.tensor(mask)
    mask = mask.float()
    if mask.dim() == 2:
        mask = mask.unsqueeze(0)
    elif mask.dim() == 3 and mask.shape[0] != 1:
        if mask.shape[-1] in (1, 3):
            mask = mask.permute(2, 0, 1)
        if mask.shape[0] != 1:
            mask = mask[:1]
    if mask.max() > 1.0:
        mask = mask / 255.0
    mask = (mask > 0.5).float()
    if mask.shape[0] != 1:
        raise ValueError(f"Expected mask [1, H, W], got {tuple(mask.shape)}")
    return mask


class RetinalProcessedDataset(Dataset):
    def __init__(
        self,
        root_dir: Path,
        split: str,
        img_size: int | None = None,
        mask_keys: tuple[str, ...] = ("manual", "manual_1", "mask", "manual_2"),
    ) -> None:
        self.root_dir = Path(root_dir)
        self.split = split
        self.files = sorted((self.root_dir / split).glob("*.pt"))
        self.img_size = img_size
        self.mask_keys = mask_keys
        if not self.files:
            raise FileNotFoundError(f"No .pt files found in {self.root_dir / split}")

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int):
        sample = torch.load(self.files[idx], map_location="cpu")
        image = sample.get("image")
        if image is None:
            raise KeyError(f"Sample {self.files[idx]} does not contain an 'image' key.")
        if not torch.is_tensor(image):
            image = torch.tensor(image)
        image = prepare_segformer_image(image)

        mask = None
        for key in self.mask_keys:
            value = sample.get(key)
            if value is not None:
                mask = value
                break
        if mask is None:
            raise KeyError(
                f"Sample {self.files[idx]} does not contain any mask key from {self.mask_keys}."
            )
        mask = prepare_binary_mask(mask)

        if self.img_size is not None:
            target_size = (self.img_size, self.img_size)
            if image.shape[-2:] != target_size:
                image = F.interpolate(
                    image.unsqueeze(0),
                    size=target_size,
                    mode="bilinear",
                    align_corners=False,
                ).squeeze(0)
            if mask.shape[-2:] != target_size:
                mask = F.interpolate(
                    mask.unsqueeze(0),
                    size=target_size,
                    mode="nearest",
                ).squeeze(0)

        return image, mask


def build_loaders(
    root: Path,
    img_size: int,
    batch_size: int,
    mask_keys: tuple[str, ...],
):
    train_ds = RetinalProcessedDataset(root, "train", img_size=img_size, mask_keys=mask_keys)
    val_ds = RetinalProcessedDataset(root, "val", img_size=img_size, mask_keys=mask_keys)
    test_ds = RetinalProcessedDataset(root, "test", img_size=img_size, mask_keys=mask_keys)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader


chase_train_ds, chase_val_ds, chase_test_ds, chase_train_loader, chase_val_loader, chase_test_loader = build_loaders(
    CHASE_ROOT,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    mask_keys=("manual_1", "manual", "mask", "manual_2"),
)

drive_train_ds, drive_val_ds, drive_test_ds, drive_train_loader, drive_val_loader, drive_test_loader = build_loaders(
    DRIVE_ROOT,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    mask_keys=("manual", "manual_1", "mask", "manual_2"),
)

print("CHASE train/val/test:", len(chase_train_ds), len(chase_val_ds), len(chase_test_ds))
print("DRIVE train/val/test:", len(drive_train_ds), len(drive_val_ds), len(drive_test_ds))

## 3. Shape Check

This verifies the model contract before training: input `[B, 3, H, W]`, output logits `[B, 1, H, W]`.


In [ ]:
shape_model = SegFormerB0(pretrained=False).to(DEVICE)
images, masks = next(iter(chase_train_loader))
with torch.no_grad():
    logits = shape_model(images.to(DEVICE))
expected_shape = (images.size(0), 1, images.size(2), images.size(3))
assert images.ndim == 4 and images.size(1) == 3, tuple(images.shape)
assert tuple(masks.shape) == expected_shape, (tuple(masks.shape), expected_shape)
assert tuple(logits.shape) == expected_shape, (tuple(logits.shape), expected_shape)
print("input:", images.shape, "masks:", masks.shape, "logits:", logits.shape)
del shape_model, images, masks, logits
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Loss, Metrics, Optimizer, Training Utilities

In [ ]:
bce_loss = torch.nn.BCEWithLogitsLoss()


def soft_dice_coef(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    intersection = (probs * targets).sum(dim=(2, 3))
    denom = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean()


def bce_dice_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return bce_loss(logits, targets) + (1.0 - soft_dice_coef(logits, targets))


def binary_dice_score(
    logits: torch.Tensor,
    targets: torch.Tensor,
    threshold: float = 0.5,
    eps: float = 1e-6,
) -> torch.Tensor:
    preds = (torch.sigmoid(logits) > threshold).float()
    intersection = (preds * targets).sum(dim=(2, 3))
    denom = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean()


def binary_iou_score(
    logits: torch.Tensor,
    targets: torch.Tensor,
    threshold: float = 0.5,
    eps: float = 1e-6,
) -> torch.Tensor:
    preds = (torch.sigmoid(logits) > threshold).float()
    intersection = (preds * targets).sum(dim=(2, 3))
    union = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()


def binary_accuracy(
    logits: torch.Tensor,
    targets: torch.Tensor,
    threshold: float = 0.5,
) -> torch.Tensor:
    preds = (torch.sigmoid(logits) > threshold).float()
    return (preds == targets).float().mean()


class EarlyStopping:
    def __init__(self, patience: int = 10, min_delta: float = 0.0, mode: str = "max") -> None:
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.counter = 0

    def step(self, score: float) -> bool:
        if self.best_score is None:
            self.best_score = score
            return False
        if self.mode == "max":
            improved = score > (self.best_score + self.min_delta)
        else:
            improved = score < (self.best_score - self.min_delta)
        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
        return self.counter >= self.patience


def get_base_model(model: torch.nn.Module) -> torch.nn.Module:
    return model.module if isinstance(model, torch.nn.DataParallel) else model


def set_encoder_trainable(model: torch.nn.Module, trainable: bool) -> None:
    base_model = get_base_model(model)
    for param in base_model.model.segformer.parameters():
        param.requires_grad = trainable


def set_all_trainable(model: torch.nn.Module) -> None:
    for param in model.parameters():
        param.requires_grad = True


def count_trainable_parameters(model: torch.nn.Module) -> tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,} / {total:,}")
    return trainable, total


def build_segformer_optimizer(
    model: torch.nn.Module,
    lr_encoder: float,
    lr_head: float,
    weight_decay: float,
) -> torch.optim.Optimizer:
    encoder_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "decode_head" in name or "classifier" in name:
            head_params.append(param)
        else:
            encoder_params.append(param)

    param_groups = []
    if encoder_params:
        param_groups.append({"params": encoder_params, "lr": lr_encoder})
    if head_params:
        param_groups.append({"params": head_params, "lr": lr_head})
    return torch.optim.AdamW(param_groups, weight_decay=weight_decay)


def amp_autocast():
    if not AMP:
        return nullcontext()
    if hasattr(torch, "amp"):
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return torch.cuda.amp.autocast(enabled=True)


def make_grad_scaler():
    if hasattr(torch, "amp"):
        return torch.amp.GradScaler("cuda", enabled=AMP)
    return torch.cuda.amp.GradScaler(enabled=AMP)


def save_checkpoint(
    model: torch.nn.Module,
    path: Path,
    epoch: int,
    metric: float,
    config: dict,
) -> None:
    base_model = get_base_model(model)
    torch.save(
        {
            "epoch": epoch,
            "best_metric": metric,
            "config": config,
            "model_state_dict": base_model.state_dict(),
        },
        path,
    )


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            value = checkpoint.get(key)
            if isinstance(value, dict):
                return value
        if all(isinstance(key, str) for key in checkpoint.keys()):
            return checkpoint
    raise ValueError("Unsupported checkpoint format")


def load_model_state_dict_safely(model: torch.nn.Module, state_dict: dict) -> None:
    candidates = [
        state_dict,
        {key.removeprefix("module."): value for key, value in state_dict.items()},
        {key.removeprefix("_orig_mod."): value for key, value in state_dict.items()},
        {key if key.startswith("model.") else f"model.{key}": value for key, value in state_dict.items()},
    ]
    errors = []
    for candidate in candidates:
        try:
            model.load_state_dict(candidate)
            return
        except RuntimeError as exc:
            errors.append(str(exc))
    raise RuntimeError(f"Could not load checkpoint. Last error: {errors[-1]}")

In [ ]:
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with amp_autocast():
            logits = model(images)
            loss = bce_dice_loss(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, threshold: float = 0.5):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_acc = 0.0
    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = bce_dice_loss(logits, masks)
        dice = binary_dice_score(logits, masks, threshold=threshold)
        iou = binary_iou_score(logits, masks, threshold=threshold)
        acc = binary_accuracy(logits, masks, threshold=threshold)
        total_loss += loss.item() * images.size(0)
        total_dice += dice.item() * images.size(0)
        total_iou += iou.item() * images.size(0)
        total_acc += acc.item() * images.size(0)

    n = len(loader.dataset)
    return total_loss / n, total_dice / n, total_iou / n, total_acc / n


def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    num_epochs: int,
    patience: int,
    checkpoint_path: Path,
    config: dict,
    threshold: float = 0.5,
):
    scaler = make_grad_scaler()
    early = EarlyStopping(patience=patience, min_delta=1e-4, mode="max")
    history = []
    best_score = -1.0

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
        val_loss, val_dice, val_iou, val_acc = evaluate(model, val_loader, threshold=threshold)
        metric = val_dice if BEST_METRIC == "dice" else val_iou

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_dice": val_dice,
                "val_iou": val_iou,
                "val_accuracy": val_acc,
            }
        )
        print(
            f"Epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} | "
            f"dice {val_dice:.4f} | iou {val_iou:.4f} | acc {val_acc:.4f}"
        )

        if metric > best_score + 1e-4:
            best_score = metric
            save_checkpoint(model, checkpoint_path, epoch, best_score, config=config)
            print(f"Saved best model to {checkpoint_path}")

        if early.step(metric):
            print("Early stopping triggered.")
            break

    return model, history

## 5. Stage 1: Source Training on CHASEDB1

Start from SegFormer-B0 pretrained weights, train on CHASEDB1, and save the best source-domain checkpoint.


In [ ]:
chase_model = SegFormerB0(pretrained=True).to(DEVICE)
set_all_trainable(chase_model)
count_trainable_parameters(chase_model)

chase_optimizer = build_segformer_optimizer(
    chase_model,
    lr_encoder=CHASE_LR_ENCODER,
    lr_head=CHASE_LR_HEAD,
    weight_decay=WEIGHT_DECAY,
)

chase_config = {
    "stage": "source_training",
    "model": "SegFormer-B0",
    "dataset": "CHASEDB1",
    "num_epochs": CHASE_NUM_EPOCHS,
    "lr_encoder": CHASE_LR_ENCODER,
    "lr_head": CHASE_LR_HEAD,
    "weight_decay": WEIGHT_DECAY,
    "patience": CHASE_PATIENCE,
    "best_metric": BEST_METRIC,
    "threshold": THRESHOLD,
}

chase_model, chase_history = train_model(
    model=chase_model,
    train_loader=chase_train_loader,
    val_loader=chase_val_loader,
    optimizer=chase_optimizer,
    num_epochs=CHASE_NUM_EPOCHS,
    patience=CHASE_PATIENCE,
    checkpoint_path=CHASE_BEST_MODEL_PATH,
    config=chase_config,
    threshold=THRESHOLD,
)

chase_history_df = pd.DataFrame(chase_history)
chase_history_df.to_csv(CHASE_HISTORY_PATH, index=False)
print(f"Saved CHASE history to {CHASE_HISTORY_PATH}")
chase_history_df.tail()

In [ ]:
best_chase_model = SegFormerB0(pretrained=False).to(DEVICE)
chase_ckpt = torch.load(CHASE_BEST_MODEL_PATH, map_location="cpu")
load_model_state_dict_safely(best_chase_model, extract_state_dict(chase_ckpt))

chase_val_loss, chase_val_dice, chase_val_iou, chase_val_acc = evaluate(
    best_chase_model, chase_val_loader, threshold=THRESHOLD
)
chase_test_loss, chase_test_dice, chase_test_iou, chase_test_acc = evaluate(
    best_chase_model, chase_test_loader, threshold=THRESHOLD
)

chase_best_epoch = int(chase_ckpt.get("epoch", chase_history_df.loc[chase_history_df["val_dice"].idxmax(), "epoch"]))
chase_results_df = pd.DataFrame(
    [
        {"split": "val", "epoch": chase_best_epoch, "loss": chase_val_loss, "dice": chase_val_dice, "iou": chase_val_iou, "accuracy": chase_val_acc},
        {"split": "test", "epoch": chase_best_epoch, "loss": chase_test_loss, "dice": chase_test_dice, "iou": chase_test_iou, "accuracy": chase_test_acc},
    ]
)
chase_results_df.to_csv(CHASE_RESULTS_PATH, index=False)
print(f"Saved CHASE results to {CHASE_RESULTS_PATH}")
chase_results_df

## 6. Stage 2: Transfer Learning / Fine-tuning on DRIVE

Load the best CHASEDB1 checkpoint into SegFormer-B0, then fine-tune on DRIVE with smaller learning rates.


In [ ]:
# TRANSFER LEARNING: LOAD CHASEDB1 SOURCE CHECKPOINT -> DRIVE
try:
    del chase_model, best_chase_model, chase_optimizer
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

drive_model = SegFormerB0(pretrained=False).to(DEVICE)
source_ckpt = torch.load(CHASE_BEST_MODEL_PATH, map_location="cpu")
load_model_state_dict_safely(drive_model, extract_state_dict(source_ckpt))
del source_ckpt
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Loaded CHASEDB1 source checkpoint from {CHASE_BEST_MODEL_PATH}")
set_all_trainable(drive_model)
count_trainable_parameters(drive_model)

drive_optimizer = build_segformer_optimizer(
    drive_model,
    lr_encoder=DRIVE_LR_ENCODER,
    lr_head=DRIVE_LR_HEAD,
    weight_decay=WEIGHT_DECAY,
)

drive_config = {
    "stage": "target_fine_tuning",
    "model": "SegFormer-B0",
    "dataset": "DRIVE",
    "source_dataset": "CHASEDB1",
    "source_checkpoint": str(CHASE_BEST_MODEL_PATH),
    "num_epochs": DRIVE_NUM_EPOCHS,
    "lr_encoder": DRIVE_LR_ENCODER,
    "lr_head": DRIVE_LR_HEAD,
    "weight_decay": WEIGHT_DECAY,
    "patience": DRIVE_PATIENCE,
    "best_metric": BEST_METRIC,
    "threshold": THRESHOLD,
}

drive_model, drive_history = train_model(
    model=drive_model,
    train_loader=drive_train_loader,
    val_loader=drive_val_loader,
    optimizer=drive_optimizer,
    num_epochs=DRIVE_NUM_EPOCHS,
    patience=DRIVE_PATIENCE,
    checkpoint_path=DRIVE_BEST_MODEL_PATH,
    config=drive_config,
    threshold=THRESHOLD,
)

drive_history_df = pd.DataFrame(drive_history)
drive_history_df.to_csv(DRIVE_HISTORY_PATH, index=False)
print(f"Saved DRIVE history to {DRIVE_HISTORY_PATH}")
drive_history_df.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(chase_history_df["epoch"], chase_history_df["train_loss"], label="CHASE train loss")
axes[0].plot(chase_history_df["epoch"], chase_history_df["val_loss"], label="CHASE val loss")
axes[0].set_title("Source Training on CHASEDB1")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(drive_history_df["epoch"], drive_history_df["train_loss"], label="DRIVE train loss")
axes[1].plot(drive_history_df["epoch"], drive_history_df["val_loss"], label="DRIVE val loss")
axes[1].set_title("Transfer Learning / Fine-tuning on DRIVE")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. Final DRIVE Evaluation and Artifact Export

In [ ]:
best_drive_model = SegFormerB0(pretrained=False).to(DEVICE)
drive_ckpt = torch.load(DRIVE_BEST_MODEL_PATH, map_location="cpu")
load_model_state_dict_safely(best_drive_model, extract_state_dict(drive_ckpt))

val_loss, val_dice, val_iou, val_acc = evaluate(best_drive_model, drive_val_loader, threshold=THRESHOLD)
test_loss, test_dice, test_iou, test_acc = evaluate(best_drive_model, drive_test_loader, threshold=THRESHOLD)

drive_best_epoch = int(drive_ckpt.get("epoch", drive_history_df.loc[drive_history_df["val_dice"].idxmax(), "epoch"]))
drive_results_df = pd.DataFrame(
    [
        {"split": "val", "epoch": drive_best_epoch, "loss": val_loss, "dice": val_dice, "iou": val_iou, "accuracy": val_acc},
        {"split": "test", "epoch": drive_best_epoch, "loss": test_loss, "dice": test_dice, "iou": test_iou, "accuracy": test_acc},
    ]
)
drive_results_df.to_csv(DRIVE_RESULTS_PATH, index=False)
shutil.copy2(DRIVE_BEST_MODEL_PATH, FINAL_BEST_MODEL_PATH)
print(f"Saved DRIVE results to {DRIVE_RESULTS_PATH}")
print(f"Copied final DRIVE checkpoint to {FINAL_BEST_MODEL_PATH}")
drive_results_df

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def denorm_image(img: torch.Tensor) -> np.ndarray:
    if img.dim() == 3 and img.shape[0] == 3:
        img_np = img.detach().cpu().numpy()
        img_np = img_np * IMAGENET_STD[:, None, None] + IMAGENET_MEAN[:, None, None]
        img_np = np.clip(img_np, 0.0, 1.0)
        return np.transpose(img_np, (1, 2, 0))
    return img.detach().cpu().numpy()


def show_predictions(model, dataset, num_samples: int = 5, threshold: float = 0.5):
    model.eval()
    DRIVE_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
    num_samples = min(num_samples, len(dataset))
    idxs = np.random.choice(len(dataset), size=num_samples, replace=False)
    saved_paths = []

    for idx in idxs:
        sample_id = int(idx)
        image, mask = dataset[idx]
        with torch.no_grad():
            logits = model(image.unsqueeze(0).to(DEVICE))
            prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
            pred = (prob > threshold).astype(np.float32)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(denorm_image(image))
        axes[0].set_title("Image")
        axes[0].axis("off")
        axes[1].imshow(mask.squeeze(0).cpu().numpy(), cmap="gray")
        axes[1].set_title("Ground Truth")
        axes[1].axis("off")
        axes[2].imshow(pred, cmap="gray")
        axes[2].set_title("Prediction")
        axes[2].axis("off")
        plt.tight_layout()

        save_path = DRIVE_PREDICTIONS_DIR / f"prediction_{sample_id:03d}.png"
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        saved_paths.append(save_path)
        plt.show()
        plt.close(fig)

    print(f"Saved {len(saved_paths)} prediction figures to {DRIVE_PREDICTIONS_DIR}")
    return saved_paths


prediction_paths = show_predictions(best_drive_model, drive_test_ds, num_samples=5, threshold=THRESHOLD)
prediction_paths

In [ ]:
summary = {
    "training_method": "Transfer Learning: CHASEDB1 source training -> DRIVE fine-tuning",
    "model": "SegFormer-B0",
    "source_dataset": "CHASEDB1",
    "target_dataset": "DRIVE",
    "chase_best_checkpoint": str(CHASE_BEST_MODEL_PATH),
    "drive_best_checkpoint": str(DRIVE_BEST_MODEL_PATH),
    "final_best_model": str(FINAL_BEST_MODEL_PATH),
    "chase_history": str(CHASE_HISTORY_PATH),
    "drive_history": str(DRIVE_HISTORY_PATH),
    "drive_results": str(DRIVE_RESULTS_PATH),
    "prediction_dir": str(DRIVE_PREDICTIONS_DIR),
    "threshold": THRESHOLD,
}

summary_path = RESULTS_DIR / "segformer_b0_chase_to_drive_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary_df = pd.DataFrame([summary])
summary_df